# 🎬 YouTube Video Pipeline on Kaggle (GPU Accelerated)

Notebook này cho phép bạn chạy toàn bộ quy trình sản xuất video 15 phút (Kokoro TTS, Whisper CUDA, Remotion Render, Kinetic Captions, Thumbnail) trên **Kaggle GPU T4 hoàn toàn miễn phí**.

### ⚠️ Các cài đặt bắt buộc trên Kaggle trước khi bấm Run:
1. Ở cột bên phải (Notebook Settings), mục **Accelerator**: Chọn **GPU T4** (hoặc GPU T4 x 2).
2. Mục **Internet**: Bật **Internet on** (Bắt buộc để tải thư viện và models).
3. Khi muốn render tự động tắt máy: Bấm nút **Save Version** -> Chọn **Save & Run All (Commit)**.

In [ ]:
# [1] Kiểm tra GPU và môi trường PyTorch CUDA
import torch
print("=== GPU Status ===")
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Settings ở góc phải -> Accelerator -> Chọn GPU T4.")

In [ ]:
# [2] Chuẩn bị mã nguồn dự án
import os

# GitHub repo & branch chứa mã nguồn và kịch bản mới nhất
GITHUB_REPO = "https://github.com/phamquanganhcode/ytb-pipeline.git"
GITHUB_BRANCH = "feat/tech-disasters"

if not os.path.exists("/kaggle/working/youtube-video-pipeline"):
    !git clone -b $GITHUB_BRANCH --single-branch $GITHUB_REPO /kaggle/working/youtube-video-pipeline
else:
    !git -C /kaggle/working/youtube-video-pipeline pull origin $GITHUB_BRANCH

# Chuyển thư mục làm việc vào repo
if os.path.exists("/kaggle/working/youtube-video-pipeline"):
    %cd /kaggle/working/youtube-video-pipeline
else:
    print("Thư mục làm việc hiện tại:", os.getcwd())

In [ ]:
# [3] Cài đặt môi trường tự động (Node.js 20, Remotion Chromium, Python packages)
!bash kaggle/setup_kaggle.sh

In [ ]:
# [4] Cấu hình API Key (Nếu bạn muốn sinh kịch bản từ --topic)
import os
os.environ["GEMINI_API_KEY"] = "" # Điền Gemini API Key nếu cần tạo kịch bản mới

In [ ]:
# [5] Chạy quy trình Render Video & Thumbnail
# Chọn kịch bản bạn muốn render:
SCRIPT = "scripts/disaster/knight-capital-2012.json"

print(f"=== Đang bắt đầu render tập phim: {SCRIPT} ===")
!python make_video.py {SCRIPT}

In [ ]:
# [6] Chuyển các file hoàn chỉnh (MP4, Thumbnail, SRT) ra mục Output để tải về
import glob
import shutil

print("=== Xuất các file kết quả ra /kaggle/working/ ===")
outputs = glob.glob("out/*.*")
for f in outputs:
    dest = os.path.join("/kaggle/working", os.path.basename(f))
    if os.path.abspath(f) != os.path.abspath(dest):
        shutil.copy(f, dest)
    print("✓ Sẵn sàng tải về:", os.path.basename(f))

print("\nHOÀN TẤT! Bạn có thể tải file MP4 và Thumbnail ở tab 'Output' bên góc phải màn hình Kaggle.")